In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("Cleaned_concussion_11_01_ver1.csv")
df

---------------------

In [ ]:
df_unclean = pd.read_csv("original_april_set.csv")
df_unclean

In [ ]:
## get first visits
#first visits


# Sort by 'record_id'
df_unclean.sort_values(by='record_id', inplace=True)

# Create a boolean mask for the first occurrence of each 'record_id'
mask = df_unclean['record_id'].ne(df_unclean['record_id'].shift(1))

# Filter the DataFrame using the mask
df_first_visits = df_unclean[mask]

In [ ]:
df_first_visits

In [ ]:
#find what record ids in common - clean set

# Create an empty list to store the values
record_id_list = []

# Iterate over the 'record_id' column and append each value to the list
for x in df['record_id']:
    record_id_list.append(x)

# Convert the list to a NumPy array if needed
record_id_array = np.array(record_id_list)

record_id_array

In [ ]:
#find what record ids in common - orig set

# Create an empty list to store the values
record_id_list_orig = []

# Iterate over the 'record_id' column and append each value to the list
for x in df_first_visits['record_id']:
    record_id_list_orig.append(x)

# Convert the list to a NumPy array if needed
record_id_array_orig = np.array(record_id_list_orig)

record_id_array_orig

In [ ]:
#find non-matches

diff = []

for val in record_id_list_orig:
    if val not in record_id_array:
        diff.append(val)

print(diff)

In [ ]:
#remove certain record ids from original

# Values to remove
diff

# Filter the DataFrame to keep rows where 'record_id' is not in the list of values to remove
df_filtered = df_first_visits[~df_first_visits['record_id'].isin(diff)]

df_filtered

In [ ]:
#filter only d2c and d2firstvisit and record_id
df_columns = df_filtered[['record_id', 'days_2_firstvisit','days_2_clearance']]
df_columns

In [ ]:
##add d2c and d2firstvisit to cleaned set to my clean set from original set

#merge only d2c and d2firstvisit to cleaned set
merged_df = df_columns.merge(df, on='record_id', how='left')
merged_df

In [ ]:
# merged_df.to_csv('11_4_merged.csv', index = False)

# print("Saved 11_4_merged.csv. ")

------------------------------------

In [ ]:
#outlier definition 1 - below 2 days
df_filtered = merged_df[(merged_df['days_2_clearance'] >= 2) & (merged_df['days_2_firstvisit'] <= 365)]

df_filtered


In [ ]:
#do one at time d2c first

df_filtered = merged_df[(merged_df['days_2_clearance'] >= 2)]
df_filtered = df_filtered[(df_filtered['days_2_firstvisit'] <= 365)]
df_filtered1 = df_filtered[(df_filtered['days_2_firstvisit'] >= 0)]

df_filtered1

In [ ]:
#do one at time d2_firstvisit first

df_filtered = merged_df[(merged_df['days_2_firstvisit'] <= 365)]
df_filtered = df_filtered[(df_filtered['days_2_clearance'] >= 1)]
df_filtered1 = df_filtered[(df_filtered['days_2_firstvisit'] >= 0)]
# df_filtered1 = df_filtered1[(df_filtered1['days_2_clearance'] <= 90)]
# regression includes 0-2 weeks, 2wk -1 month , 1mth - 2 mth - 2mth - 3mth,  90 or 90+ days -> classification
# see # people in each bin

df_filtered1

--------------

In [ ]:
#outlier definition 2 - z score
#determine z score of each row d2c
#keeps rows z scores -3 and above
#by z score - A Z-Score is a statistical measurement of a score's relationship to the mean in a group of scores.

def remove_left_outliers_zscore(df, columns, threshold=3):
  """
  Removes outliers on the left side of the specified columns using the Z-score method.

  Args:
    df: The DataFrame.
    columns: A list of column names to remove outliers from.
    threshold: The Z-score threshold for identifying outliers.

  Returns:
    The DataFrame with left-side outliers removed.
  """

  for col in columns:
    z_scores = (df[col] - df[col].mean()) / df[col].std()
    df = df[z_scores >= -threshold]

  return df

In [ ]:
# Remove left-side outliers from specified columns with a Z-score threshold of 3
df_filtered1 = remove_left_outliers_zscore(merged_df, ['days_2_clearance'], threshold=3)

df_filtered1

In [ ]:
#determine value at x std for days 2 clearance

import numpy as np

def calculate_values_at_3std(data):
  """
  Calculates the values that are 3 standard deviations away from the mean.

  Args:
    data: A list or NumPy array of numerical data.

  Returns:
    A tuple containing the lower and upper values.
  """

  mean = np.mean(data)
  std_dev = np.std(data)

  lower_value = mean - (3 * std_dev)

  return lower_value

In [ ]:
# Remove left-side outliers from specified columns with a Z-score threshold of 3
df_filtered2 = remove_left_outliers_zscore(merged_df, ['days_2_clearance'], threshold=2)

df_filtered2

In [ ]:
# Remove left-side outliers from specified columns with a Z-score threshold of 3
df_filtered3 = remove_left_outliers_zscore(merged_df, ['days_2_clearance'], threshold=1)

df_filtered3

In [ ]:
## days_2_firstvisit

#outlier definition 2 - z score
#determine z score of each row d2c
#keeps rows z scores -3 and above
#by z score - A Z-Score is a statistical measurement of a score's relationship to the mean in a group of scores.

def remove_right_outliers_zscore(df, columns, threshold=3):
  """
  Removes outliers on the left side of the specified columns using the Z-score method.

  Args:
    df: The DataFrame.
    columns: A list of column names to remove outliers from.
    threshold: The Z-score threshold for identifying outliers.

  Returns:
    The DataFrame with left-side outliers removed.
  """

  for col in columns:
    z_scores = (df[col] - df[col].mean()) / df[col].std()
    df = df[z_scores <= threshold]

  return df

In [ ]:
# Remove right-side outliers from specified columns with a Z-score threshold of 3
df_filtered1_1 = remove_right_outliers_zscore(df_filtered1, ['days_2_firstvisit'], threshold=3)

df_filtered1_1

## Reverse order filter - z score

In [ ]:
# reverse order - days first visit then do days 2 clearance




df_filtered1_1 = remove_right_outliers_zscore(merged_df, ['days_2_firstvisit'], threshold=3)

df_filtered1_1

In [ ]:
df_filtered1_2 = remove_left_outliers_zscore(df_filtered1_1, ['days_2_clearance'], threshold=3)
df_filtered1_2

----------------

In [ ]:
#remove q3 first

import pandas as pd

def remove_above_q3(df, column):
  """
  Removes rows from the DataFrame where the specified column value is above the third quartile (Q3).

  Args:
    df: The DataFrame.
    column: The column name to check for values above Q3.

  Returns:
    The DataFrame with rows above Q3 removed.
  """

  Q3 = df[column].quantile(0.75)
  df = df[df[column] <= Q3]
  return df

# Assuming 'merged_df' is your DataFrame
df_filtered = remove_above_q3(merged_df, 'days_2_firstvisit')
df_filtered

In [ ]:
import pandas as pd

def remove_below_q1(df, column):
  """
  Removes rows from the DataFrame where the specified column value is below the first quartile (Q1).

  Args:
    df: The DataFrame.
    column: The column name to check for values below Q1.

  Returns:
    The DataFrame with rows below Q1 removed.
  """

  Q1 = df[column].quantile(0.25)
  df = df[df[column] >= Q1]
  return df

# Assuming 'merged_df' is your DataFrame
df_filtered = remove_below_q1(merged_df, 'days_2_clearance')
df_filtered

In [ ]:
clean_data(df_filtered)